In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout, SimpleRNN
from tensorflow.keras.metrics import SparseTopKCategoricalAccuracy
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.activations import leaky_relu
import keras_tuner as kt

In [79]:
STEP = 7

In [12]:
X = np.loadtxt('../data/features/all_step_features+.csv', delimiter=',')
y = np.loadtxt('../data/features/all_step_label+.csv', delimiter=',')

print('X shape:', X.shape)
print('y shape:', y.shape)

X shape: (2954, 782)
y shape: (2954,)


In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=5)

print('X Train shape:', X_train.shape)
print('X Test shape:', X_test.shape)
print('y Train shape:', y_train.shape)
print('y Test shape:', y_test.shape)

X Train shape: (2658, 782)
X Test shape: (296, 782)
y Train shape: (2658,)
y Test shape: (296,)


In [15]:
ann = Sequential([
  Input(shape=(X_train.shape[1],)),
  Dense(512, activation='relu'),
  Dense(256, activation='relu'),
  Dense(128, activation='softmax')
])
ann.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_5 (Dense)                 │ (None, 512)            │       400,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 128)            │        32,896 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 565,120 (2.16 MB)

 Trainable params: 565,120 (2.16 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
ann.compile(loss='sparse_categorical_crossentropy', optimizer=Adam(), metrics=['accuracy', SparseTopKCategoricalAccuracy(k=5)])
ann.fit(X_train, y_train, epochs=100, validation_split=0.2)

Epoch 1/100
67/67 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.0295 - loss: 4.3484 - sparse_top_k_categorical_accuracy: 0.1333 - val_accuracy: 0.0357 - val_loss: 3.9776 - val_sparse_top_k_categorical_accuracy: 0.1974
Epoch 2/100
67/67 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.0665 - loss: 3.8218 - sparse_top_k_categorical_accuracy: 0.2674 - val_accuracy: 0.0564 - val_loss: 3.8528 - val_sparse_top_k_categorical_accuracy: 0.2331
Epoch 3/100
67/67 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.0931 - loss: 3.6271 - sparse_top_k_categorical_accuracy: 0.3261 - val_accuracy: 0.0545 - val_loss: 3.8071 - val_sparse_top_k_categorical_accuracy: 0.2500
Epoch 4/100
67/67 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.1212 - loss: 3.4593 - sparse_top_k_categorical_accuracy: 0.3762 - val_accuracy: 0.0489 - val_loss: 3.7980 - val_sparse_top_k_categorical_accuracy: 0.2594
Epoch 5/100
67/67 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.1363 - loss: 3.2449 - sparse_top_k_categorical_accuracy:

In [17]:
ann.evaluate(X_test, y_test)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0760 - loss: 10.9441 - sparse_top_k_categorical_accuracy: 0.3295 


[11.2422513961792, 0.07094594836235046, 0.2939189076423645]

### Tuning Hyperparameters

In [19]:
def build_model(hp):
  model = Sequential()
  model.add(Input(shape=(782,)))

  # Hidden Layer 1
  model.add(Dense(
    units=hp.Choice('units1', [128, 256, 512, 1024, 2048]),
    activation='relu',
  ))

  # Dropout Layer 1 (Optional)
  if hp.Boolean('use_dropout1_layer'):
    model.add(Dropout(hp.Float('dropout1', min_value=0.1, max_value=0.5, step=0.1)))

  # Hidden Layer 2 (Optional)
  if hp.Boolean('use_hidden_layer2'):
    model.add(Dense(
      units=hp.Choice('units2', [128, 256, 512, 1024, 2048]),
      activation='relu'
    ))

  # Dropout Layer 2 (Optional)
  if hp.Boolean('use_dropout2_layer'):
    model.add(Dropout(hp.Float('dropout2', min_value=0.1, max_value=0.5, step=0.1)))

  # Output Layer
  model.add(Dense(128, activation='softmax'))

  # Optimizer and Learning Rate
  lr = hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])
  optimizer = Adam(learning_rate=lr)

  model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', SparseTopKCategoricalAccuracy(k=5)]
  )

  return model

In [20]:
tuner = kt.RandomSearch(
  build_model,
  objective='val_accuracy',
  max_trials=20,
  executions_per_trial=1,
  directory='../tuning_dir',
  project_name=f'_allstep+'
)

tuner.search(X_train, y_train, epochs=100, validation_split=0.1)

Trial 20 Complete [00h 03m 11s]
val_accuracy: 0.10902255773544312

Best val_accuracy So Far: 0.1315789520740509
Total elapsed time: 00h 56m 11s


In [21]:
best_model = tuner.get_best_models()[0]
best_hps = tuner.get_best_hyperparameters()[0]

print("Best Hyperparameters:")
print(f"Units1: {best_hps['units1']}")
print(f"Use Dropout1 Layer: {best_hps['use_dropout1_layer']}")
if best_hps['use_dropout1_layer']:
    print(f"Dropout1: {best_hps['dropout1']}")
print(f"Use Second Layer: {best_hps['use_hidden_layer2']}")
if best_hps['use_hidden_layer2']:
    print(f"Units2: {best_hps['units2']}")
print(f"Use Dropout2 Layer: {best_hps['use_dropout2_layer']}")
if best_hps['use_dropout2_layer']:
    print(f"Dropout2: {best_hps['dropout2']}")
print(f"Learning Rate: {best_hps['learning_rate']}")

Best Hyperparameters:
Units1: 512
Use Dropout1 Layer: False
Use Second Layer: False
Use Dropout2 Layer: False
Learning Rate: 0.01


In [22]:
best_model.evaluate(X_test, y_test)

10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.0952 - loss: 15.0393 - sparse_top_k_categorical_accuracy: 0.2677  


[15.398221015930176, 0.09121621400117874, 0.24662162363529205]

In [23]:
best_trial = tuner.oracle.get_best_trials(num_trials=1)[0]

print("Best trial ID:", best_trial.trial_id)
print("Best val_accuracy:", best_trial.score)
print("Best hyperparameters:", best_trial.hyperparameters.values)


Best trial ID: 03
Best val_accuracy: 0.1315789520740509
Best hyperparameters: {'units1': 512, 'use_dropout1_layer': False, 'use_hidden_layer2': False, 'use_dropout2_layer': False, 'learning_rate': 0.01, 'units2': 512, 'dropout2': 0.2}
